# 2. Basics of Optimization

## Part C Multiobjective optimization

We consider a two-layer structure of materials A and B with variable thicknesses $\ell_A$ and $\ell_B$. The total thickness $L = \ell_A + \ell_B$ is no longer fixed.
Your goal is to simultaneously:
* Minimize the total thickness $L$
* Maximize the acoustic absorption

Your Task:
Adjust the trade-off weight $w \in [0, 1]$ in the code cell below. 
Setting $w \to 0$ prioritizes absorption performance regardless of sample thickness.
Setting $w \to 1$ penalizes thickness most heavily, favoring compact designs at the expense of acoustic performance.
Play around with the value of `WEIGHT` to find a structure that achieves sufficient absorption while keeping the total thickness minimal - until the trade-off is optimized to your satisfaction.



In [ ]:
# import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from optimization_utils import *

In [ ]:
# Select materials
matA = MATERIALS["acoustic_foam"]
matB = MATERIALS["melamine_foam"]

# initial guess for the thicknesses of matA and matB
x0 = [L / 4, L / 4] 

bounds = [(0.0, L), (0.0, L)]

# Inequality constraint: L - (l_A + l_B) >= 0 (total length <= L)
constraints = [{'type': 'ineq', 'fun': lambda x: L - (x[0] + x[1])}]


In [ ]:

# TODO adjust and play around to get better results
WEIGHT = 0.05 # weight balancing thickness penalty vs absorption performance 

def objective_scalarized(x):
    l_A, l_B = x
    if (l_A + l_B) < 1e-6:
        return 0.0
    
    reflection, absorption = compute_spectrum([l_A, l_B], [matA, matB])
    obj = -np.mean(absorption) + WEIGHT * ((l_A + l_B) / L)
    # obj = np.mean(np.abs(reflection)) + WEIGHT * ((l_A + l_B) / L)
    # obj = -np.mean(absorption[freqs<300]) + WEIGHT * ((l_A + l_B) / L)
    # obj = -np.max(absorption) + WEIGHT * ((l_A + l_B) / L)            

    return obj

res_scipy = minimize(
    objective_scalarized,
    x0,
    method='SLSQP',
    bounds=bounds,
    # constraints=constraints
)

opt_lA, opt_lB = res_scipy.x
tot_L = opt_lA + opt_lB

print(f"{matA.name} (A): {opt_lA * 1000:.2f} mm")
print(f"{matB.name} (B): {opt_lB * 1000:.2f} mm")
print(f"Total Length: {tot_L * 1000:.2f} mm")

In [ ]:
# Reflection and absorption for optimal case and single-material baselines
r_opt, abs_opt = compute_spectrum([opt_lA, opt_lB], [matA, matB])
r_matA, abs_matA = compute_spectrum([opt_lA + opt_lB], [matA])
r_matB, abs_matB = compute_spectrum([opt_lA + opt_lB], [matB])

# Plot results
plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.plot(freqs, np.abs(r_opt), 'k-', linewidth=2, label=f'Part C: optimized ({tot_L*1000:.1f} mm total)')
plt.plot(freqs, np.abs(r_matA), '--', label=f'Only {matA.name}')
plt.plot(freqs, np.abs(r_matB), '--', label=f'Only {matB.name}')
setup_r_axis()

plt.subplot(122)
plt.plot(freqs, abs_opt, 'k-', linewidth=2, label=f'Part C: optimized ({tot_L*1000:.1f} mm total)')
plt.plot(freqs, abs_matA, '--', label=f'Only {matA.name}')
plt.plot(freqs, abs_matB, '--', label=f'Only {matB.name}')
setup_abs_axis()

plt.show()